# Incident Response Runbook: Initial Access - Phishing

**Tactic:** Initial Access
**Technique:** Phishing (T1566)
**Severity:** HIGH

## Overview

This runbook provides step-by-step incident response procedures for Phishing activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

# Add project root to path for imports
sys.path.append(os.path.join(os.path.dirname(__file__), '..', '..'))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Initialize integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

detection_time = datetime.now().isoformat()
affected_systems = []
phishing_indicators = []
incident_id = None

# Query Splunk for phishing indicators
print(f"\n[DETECTION] Querying Splunk for phishing activities...")
try:
    # Email-based phishing detection
    email_phish_query = """
    index=* sourcetype=email sourcetype=exchange sourcetype=o365
    | search (phishing OR "suspicious email" OR "malicious link" OR "credential theft")
    | eval risk_score = case(
        match(subject, "(password|credential|login|account)"), 9,
        match(body, "urgent|immediate|action required"), 8,
        match(sender, "\.(ru|cn|tk|ml|ga)$"), 8,
        match(_raw, "suspicious attachment"), 9,
        match(_raw, "clicked malicious link"), 10,
        1==1, 6
    )
    | where risk_score >= 7
    | stats count by sender, recipient, subject, risk_score, attachment_name
    | sort -risk_score
    """

    email_results = splunk.search(email_phish_query, timeframe="-24h")
    if email_results:
        print(f"   Found {len(email_results)} suspicious email activities")
        for result in email_results[:10]:  # Top 10
            phishing_indicators.append({
                'type': 'email_phishing',
                'sender': result.get('sender'),
                'recipient': result.get('recipient'),
                'subject': result.get('subject'),
                'attachment': result.get('attachment_name'),
                'risk_score': result.get('risk_score')
            })

    # Credential phishing detection
    cred_phish_query = """
    index=* sourcetype=web_proxy sourcetype=vpn sourcetype=authentication
    | search ("fake login" OR "phishing site" OR "credential harvesting")
    | eval risk_score = case(
        match(url, "login.*fake|fake.*login"), 10,
        match(url, "\.(ru|cn|tk|ml)"), 9,
        match(user_agent, "suspicious"), 7,
        match(_raw, "credential entered"), 8,
        1==1, 5
    )
    | where risk_score >= 7
    | stats count by src_ip, url, user, risk_score
    | sort -risk_score
    """

    cred_results = splunk.search(cred_phish_query, timeframe="-24h")
    if cred_results:
        print(f"   Found {len(cred_results)} credential phishing attempts")
        for result in cred_results[:10]:  # Top 10
            phishing_indicators.append({
                'type': 'credential_phishing',
                'src_ip': result.get('src_ip'),
                'url': result.get('url'),
                'user': result.get('user'),
                'risk_score': result.get('risk_score')
            })

    # Spear-phishing detection
    spear_phish_query = """
    index=* sourcetype=email sourcetype=exchange
    | search ("targeted email" OR "spear phishing" OR "whaling")
    | eval risk_score = case(
        match(recipient, "executive|ceo|cfo|cto"), 10,
        match(subject, "invoice|payment|contract"), 8,
        match(attachment, "\.docm|\.xlsm|\.pptm"), 9,
        match(_raw, "macro enabled"), 8,
        1==1, 6
    )
    | where risk_score >= 7
    | stats count by sender, recipient, subject, attachment_name, risk_score
    | sort -risk_score
    """

    spear_results = splunk.search(spear_phish_query, timeframe="-24h")
    if spear_results:
        print(f"   Found {len(spear_results)} spear-phishing attempts")
        for result in spear_results[:10]:  # Top 10
            phishing_indicators.append({
                'type': 'spear_phishing',
                'sender': result.get('sender'),
                'recipient': result.get('recipient'),
                'subject': result.get('subject'),
                'attachment': result.get('attachment_name'),
                'risk_score': result.get('risk_score')
            })

except Exception as e:
    print(f"   Splunk query failed: {e}")

# Get unique affected systems/users
unique_recipients = set()
unique_ips = set()
for indicator in phishing_indicators:
    if indicator.get('recipient'):
        unique_recipients.add(indicator['recipient'])
    if indicator.get('src_ip'):
        unique_ips.add(indicator['src_ip'])

# Check CrowdStrike for related endpoint activity
print(f"\n[DETECTION] Checking CrowdStrike for related endpoint activity...")
for ip in unique_ips:
    try:
        cs_alerts = crowdstrike.get_alerts_by_ip(ip)
        if cs_alerts:
            for alert in cs_alerts[:5]:  # Top 5 alerts per IP
                affected_systems.append({
                    'ip': ip,
                    'alerts': len(cs_alerts),
                    'device_id': alert.get('device_id'),
                    'hostname': alert.get('hostname')
                })
            print(f"   Found {len(cs_alerts)} CrowdStrike alerts for IP {ip}")
    except Exception as e:
        print(f"   CrowdStrike check failed for {ip}: {e}")

# Enrich with threat intelligence
print(f"\n[INTEL] Enriching with threat intelligence...")
for indicator in phishing_indicators[:5]:  # Check top 5
    try:
        if indicator.get('sender'):
            # Extract domain from sender
            sender_domain = indicator['sender'].split('@')[-1] if '@' in indicator['sender'] else indicator['sender']
            ti_data = misp.lookup_domain(sender_domain)
            if ti_data:
                indicator['threat_intel'] = ti_data
                print(f"   Threat intel found for sender domain: {sender_domain}")
        elif indicator.get('url'):
            ti_data = misp.lookup_url(indicator['url'])
            if ti_data:
                indicator['threat_intel'] = ti_data
                print(f"   Threat intel found for URL: {indicator['url']}")
    except Exception as e:
        print(f"   Threat intelligence lookup failed: {e}")

# Create IRIS case
print(f"\n[CASE] Creating IRIS incident case...")
try:
    case_data = {
        'title': f'Phishing Attack - {len(phishing_indicators)} indicators detected',
        'description': f'Detected phishing activities affecting {len(unique_recipients)} users and {len(unique_ips)} source IPs',
        'severity': 'HIGH',
        'tactic': 'Initial Access',
        'technique': 'Phishing (T1566)',
        'indicators': phishing_indicators[:10],  # Top 10 indicators
        'detection_time': detection_time
    }
    incident_id = iris.create_case(case_data)
    print(f"   Created IRIS case: {incident_id}")
except Exception as e:
    print(f"   IRIS case creation failed: {e}")

print(f"\n Detection complete:")
print(f"  - {len(phishing_indicators)} phishing indicators detected")
print(f"  - {len(unique_recipients)} affected users identified")
print(f"  - {len(unique_ips)} suspicious source IPs found")
print(f"  - Threat intelligence enriched for {len([i for i in phishing_indicators if i.get('threat_intel')])} indicators")
print(f"  - IRIS case created: {incident_id}")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

containment_time = datetime.now().isoformat()
isolated_systems = []
blocked_senders = []
blocked_domains = []
disabled_accounts = []

# Isolate affected systems
print(f"\n[ACTION] Isolating affected systems...")
for system in affected_systems:
    try:
        if system.get('device_id'):
            isolation_result = crowdstrike.isolate_host(system['device_id'])
            if isolation_result.get('status') == 'isolated':
                isolated_systems.append(system.get('hostname', system.get('ip', 'unknown')))
                print(f"   Isolated system: {system.get('hostname', system.get('ip', 'unknown'))}")
    except Exception as e:
        print(f"   System isolation failed for {system.get('hostname', system.get('ip', 'unknown'))}: {e}")

# Block phishing sender addresses
print(f"\n[ACTION] Blocking phishing sender addresses...")
unique_senders = set()
for indicator in phishing_indicators:
    if indicator.get('sender'):
        unique_senders.add(indicator['sender'])

for sender in unique_senders:
    try:
        block_result = shuffle.block_email_sender(sender)
        if block_result.get('status') == 'blocked':
            blocked_senders.append(sender)
            print(f"   Blocked phishing sender: {sender}")
    except Exception as e:
        print(f"   Sender blocking failed for {sender}: {e}")

# Block phishing domains
print(f"\n[ACTION] Blocking phishing domains...")
unique_domains = set()
for indicator in phishing_indicators:
    if indicator.get('sender') and '@' in indicator['sender']:
        domain = indicator['sender'].split('@')[-1]
        unique_domains.add(domain)
    elif indicator.get('url'):
        # Extract domain from URL
        import urllib.parse
        parsed = urllib.parse.urlparse(indicator['url'])
        if parsed.netloc:
            unique_domains.add(parsed.netloc)

for domain in unique_domains:
    try:
        block_result = shuffle.block_domain(domain)
        if block_result.get('status') == 'blocked':
            blocked_domains.append(domain)
            print(f"   Blocked phishing domain: {domain}")
    except Exception as e:
        print(f"   Domain blocking failed for {domain}: {e}")

# Disable compromised user accounts
print(f"\n[ACTION] Disabling compromised user accounts...")
unique_users = set()
for indicator in phishing_indicators:
    if indicator.get('recipient'):
        unique_users.add(indicator['recipient'])

for user in unique_users:
    try:
        disable_result = shuffle.disable_user_account(user)
        if disable_result.get('status') == 'disabled':
            disabled_accounts.append(user)
            print(f"   Disabled account: {user}")
    except Exception as e:
        print(f"   Account disable failed for {user}: {e}")

# Enable enhanced phishing monitoring
print(f"\n[ACTION] Enabling enhanced phishing monitoring...")
monitoring_rules = []
try:
    # Enable CrowdStrike phishing monitoring
    cs_monitoring = crowdstrike.enable_enhanced_monitoring('phishing_attack')
    if cs_monitoring.get('status') == 'enabled':
        monitoring_rules.append('CrowdStrike phishing monitoring')
        print(f"   Enabled CrowdStrike phishing attack monitoring")

    # Enable Splunk phishing correlation rules
    splunk_monitoring = splunk.enable_correlation_rule('phishing_attack')
    if splunk_monitoring.get('status') == 'enabled':
        monitoring_rules.append('Splunk phishing correlation')
        print(f"   Enabled Splunk phishing attack correlation rules")
except Exception as e:
    print(f"   Enhanced monitoring setup failed: {e}")

# Update IRIS case with containment actions
print(f"\n[ACTION] Updating IRIS case with containment actions...")
try:
    containment_summary = {
        'isolated_systems': len(isolated_systems),
        'blocked_senders': len(blocked_senders),
        'blocked_domains': len(blocked_domains),
        'disabled_accounts': len(disabled_accounts),
        'monitoring_enabled': len(monitoring_rules),
        'containment_time': containment_time
    }
    iris.update_case(incident_id, 'containment_complete', containment_summary)
    print(f"   Updated IRIS case {incident_id} with containment results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

print(f"\n Containment complete:")
print(f"  - {len(isolated_systems)} systems isolated")
print(f"  - {len(blocked_senders)} phishing senders blocked")
print(f"  - {len(blocked_domains)} phishing domains blocked")
print(f"  - {len(disabled_accounts)} accounts disabled")
print(f"  - {len(monitoring_rules)} enhanced monitoring rules enabled")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

eradication_time = datetime.now().isoformat()
terminated_processes = []
removed_attachments = []
cleaned_emails = []
removed_payloads = []

# Terminate phishing-related processes
print(f"\n[ACTION] Terminating phishing-related processes...")
for system in affected_systems:
    try:
        if system.get('device_id'):
            processes = crowdstrike.get_suspicious_processes(system['device_id'], 'phishing')
            for proc in processes:
                if proc.get('process_name') in ['phishing.exe', 'malware_dropper.exe', 'credential_harvester.exe', 'keylogger.exe']:
                    term_result = crowdstrike.terminate_process(system['device_id'], proc['process_id'])
                    if term_result.get('status') == 'terminated':
                        terminated_processes.append(f"{system.get('hostname', 'unknown')}:{proc['process_name']}")
                        print(f"   Terminated process: {proc['process_name']} on {system.get('hostname', 'unknown')}")
    except Exception as e:
        print(f"   Process termination failed: {e}")

# Remove malicious email attachments
print(f"\n[ACTION] Removing malicious email attachments...")
try:
    for indicator in phishing_indicators:
        if indicator.get('attachment'):
            attachment_result = splunk.quarantine_attachment(indicator['attachment'])
            if attachment_result.get('status') == 'quarantined':
                removed_attachments.append(indicator['attachment'])
                print(f"   Quarantined malicious attachment: {indicator['attachment']}")
except Exception as e:
    print(f"   Attachment removal failed: {e}")

# Clean up phishing emails
print(f"\n[ACTION] Cleaning up phishing emails...")
try:
    email_cleanup = splunk.clean_phishing_emails()
    if email_cleanup.get('status') == 'cleaned':
        cleaned_emails.append(f"Emails cleaned: {email_cleanup.get('count', 0)} messages")
        print(f"   Cleaned up {email_cleanup.get('count', 0)} phishing emails")
except Exception as e:
    print(f"   Email cleanup failed: {e}")

# Remove phishing payloads and malware
print(f"\n[ACTION] Removing phishing payloads and malware...")
for system in affected_systems:
    try:
        if system.get('device_id'):
            payloads = crowdstrike.get_phishing_payloads(system['device_id'])
            for payload_path in payloads:
                remove_result = crowdstrike.remove_file(system['device_id'], payload_path)
                if remove_result.get('status') == 'removed':
                    removed_payloads.append(f"{system.get('hostname', 'unknown')}:{payload_path}")
                    print(f"   Removed phishing payload: {payload_path} from {system.get('hostname', 'unknown')}")
    except Exception as e:
        print(f"   Payload removal failed for {system.get('hostname', 'unknown')}: {e}")

# Reset compromised credentials
print(f"\n[ACTION] Resetting compromised credentials...")
try:
    unique_users = set()
    for indicator in phishing_indicators:
        if indicator.get('recipient'):
            unique_users.add(indicator['recipient'])

    for user in unique_users:
        reset_result = shuffle.reset_user_password(user)
        if reset_result.get('status') == 'reset':
            print(f"   Reset password for compromised user: {user}")
except Exception as e:
    print(f"   Password reset failed: {e}")

# Update threat intelligence
print(f"\n[ACTION] Updating threat intelligence...")
try:
    eradication_iocs = {
        'phishing_senders': [i.get('sender') for i in phishing_indicators if i.get('sender')],
        'phishing_domains': list(unique_domains),
        'phishing_urls': [i.get('url') for i in phishing_indicators if i.get('url')],
        'compromised_users': [i.get('recipient') for i in phishing_indicators if i.get('recipient')],
        'malicious_attachments': [i.get('attachment') for i in phishing_indicators if i.get('attachment')]
    }
    misp.share_iocs(eradication_iocs, 'phishing_attack_eradicated')
    print(f"   Shared eradication IOCs with MISP threat intelligence")
except Exception as e:
    print(f"   Threat intelligence update failed: {e}")

# Update IRIS case with eradication actions
print(f"\n[ACTION] Updating IRIS case with eradication actions...")
try:
    eradication_summary = {
        'terminated_processes': len(terminated_processes),
        'removed_attachments': len(removed_attachments),
        'cleaned_emails': len(cleaned_emails),
        'removed_payloads': len(removed_payloads),
        'eradication_time': eradication_time
    }
    iris.update_case(incident_id, 'eradication_complete', eradication_summary)
    print(f"   Updated IRIS case {incident_id} with eradication results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

print(f"\n Eradication complete:")
print(f"  - {len(terminated_processes)} malicious processes terminated")
print(f"  - {len(removed_attachments)} malicious attachments quarantined")
print(f"  - {len(cleaned_emails)} phishing emails cleaned")
print(f"  - {len(removed_payloads)} phishing payloads removed")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

recovery_time = datetime.now().isoformat()
restored_systems = []
recreated_accounts = []
restored_services = []

# Restore isolated systems
print(f"\n[ACTION] Restoring isolated systems...")
for system in affected_systems:
    try:
        if system.get('device_id') and system.get('hostname') in isolated_systems:
            restore_result = crowdstrike.restore_host(system['device_id'])
            if restore_result.get('status') == 'restored':
                restored_systems.append(system['hostname'])
                print(f"   Restored system: {system['hostname']}")
    except Exception as e:
        print(f"   System restoration failed for {system.get('hostname', 'unknown')}: {e}")

# Recreate disabled user accounts
print(f"\n[ACTION] Recreating disabled user accounts...")
for user in disabled_accounts:
    try:
        recreate_result = shuffle.recreate_user_account(user)
        if recreate_result.get('status') == 'recreated':
            recreated_accounts.append(user)
            print(f"   Recreated account: {user}")
    except Exception as e:
        print(f"   Account recreation failed for {user}: {e}")

# Restore affected email services
print(f"\n[ACTION] Restoring affected email services...")
try:
    # Restore email filtering
    email_restore = shuffle.restore_email_service()
    if email_restore.get('status') == 'restored':
        restored_services.append('Email service')
        print(f"   Restored email service")

    # Restore web proxy services
    proxy_restore = shuffle.restore_web_proxy()
    if proxy_restore.get('status') == 'restored':
        restored_services.append('Web proxy')
        print(f"   Restored web proxy service")

    # Restore authentication services
    auth_restore = shuffle.restore_authentication_service()
    if auth_restore.get('status') == 'restored':
        restored_services.append('Authentication service')
        print(f"   Restored authentication service")
except Exception as e:
    print(f"   Service restoration failed: {e}")

# Validate system integrity
print(f"\n[ACTION] Validating system integrity...")
integrity_checks = []
try:
    for system in affected_systems:
        if system.get('device_id'):
            integrity_result = crowdstrike.validate_system_integrity(system['device_id'])
            if integrity_result.get('status') == 'valid':
                integrity_checks.append(system.get('hostname', system.get('ip', 'unknown')))
                print(f"   System integrity validated: {system.get('hostname', system.get('ip', 'unknown'))}")
            else:
                print(f"   System integrity issues detected: {system.get('hostname', system.get('ip', 'unknown'))}")
except Exception as e:
    print(f"   Integrity validation failed: {e}")

# Re-enable standard monitoring
print(f"\n[ACTION] Re-enabling standard monitoring...")
try:
    # Restore normal CrowdStrike monitoring
    cs_restore = crowdstrike.restore_standard_monitoring()
    if cs_restore.get('status') == 'restored':
        print(f"   Restored standard CrowdStrike monitoring")

    # Restore normal Splunk correlation rules
    splunk_restore = splunk.restore_standard_correlation()
    if splunk_restore.get('status') == 'restored':
        print(f"   Restored standard Splunk correlation rules")
except Exception as e:
    print(f"   Monitoring restoration failed: {e}")

# Update IRIS case with recovery actions
print(f"\n[ACTION] Updating IRIS case with recovery actions...")
try:
    recovery_summary = {
        'restored_systems': len(restored_systems),
        'recreated_accounts': len(recreated_accounts),
        'restored_services': len(restored_services),
        'integrity_validated': len(integrity_checks),
        'recovery_time': recovery_time
    }
    iris.update_case(incident_id, 'recovery_complete', recovery_summary)
    print(f"   Updated IRIS case {incident_id} with recovery results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

print(f"\n Recovery complete:")
print(f"  - {len(restored_systems)} systems restored")
print(f"  - {len(recreated_accounts)} accounts recreated")
print(f"  - {len(restored_services)} services restored")
print(f"  - {len(integrity_checks)} systems validated")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident
print("\n" + "=" * 60)
print("STEP 5: Post-Incident")
print("=" * 60)

post_incident_time = datetime.now().isoformat()

# Generate incident report
print(f"\n[ACTION] Generating incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'technique': 'Phishing (T1566)',
        'detection_time': detection_time,
        'containment_time': containment_time,
        'eradication_time': eradication_time,
        'recovery_time': recovery_time,
        'post_incident_time': post_incident_time,
        'affected_systems': len(affected_systems),
        'phishing_indicators': len(phishing_indicators),
        'unique_recipients': len(unique_recipients),
        'unique_ips': len(unique_ips),
        'isolated_systems': len(isolated_systems),
        'blocked_senders': len(blocked_senders),
        'blocked_domains': len(blocked_domains),
        'disabled_accounts': len(disabled_accounts),
        'terminated_processes': len(terminated_processes),
        'removed_attachments': len(removed_attachments),
        'cleaned_emails': len(cleaned_emails),
        'removed_payloads': len(removed_payloads),
        'restored_systems': len(restored_systems),
        'recreated_accounts': len(recreated_accounts),
        'restored_services': len(restored_services),
        'integrity_validated': len(integrity_checks),
        'threat_intelligence_shared': True,
        'lessons_learned': [
            'Enhanced email filtering and authentication needed',
            'User training on phishing recognition required',
            'Improved monitoring for suspicious email patterns',
            'Regular threat intelligence updates for phishing IOCs',
            'Multi-factor authentication enforcement needed'
        ]
    }

    # Save report to IRIS
    iris.save_incident_report(incident_id, incident_report)
    print(f"   Incident report saved to IRIS case {incident_id}")

    # Share anonymized report with MISP
    misp.share_incident_report(incident_report, 'phishing_attack_response')
    print(f"   Anonymized incident report shared with MISP community")
except Exception as e:
    print(f"   Report generation failed: {e}")

# Conduct lessons learned session
print(f"\n[ACTION] Conducting lessons learned analysis...")
try:
    lessons_learned = {
        'what_went_well': [
            'Rapid detection through email monitoring',
            'Effective containment via sender/domain blocking',
            'Comprehensive threat intelligence integration',
            'Automated response workflows functioned correctly'
        ],
        'what_could_improve': [
            'Earlier detection of sophisticated phishing campaigns',
            'Better user awareness and training programs',
            'Enhanced email gateway security controls',
            'More granular monitoring of user behavior',
            'Improved integration between email and endpoint security'
        ],
        'preventive_measures': [
            'Implement DMARC/SPF/DKIM for email authentication',
            'Deploy advanced email filtering and sandboxing',
            'Conduct regular phishing awareness training',
            'Enable multi-factor authentication organization-wide',
            'Implement zero-trust network access controls',
            'Regular security assessments of email systems'
        ]
    }

    iris.add_lessons_learned(incident_id, lessons_learned)
    print(f"   Lessons learned documented in IRIS case {incident_id}")
except Exception as e:
    print(f"   Lessons learned analysis failed: {e}")

# Update security controls
print(f"\n[ACTION] Updating security controls...")
try:
    # Update Splunk correlation rules
    splunk.update_phishing_rules(incident_report)
    print(f"   Updated Splunk phishing detection rules")

    # Update CrowdStrike policies
    crowdstrike.update_phishing_policies(incident_report)
    print(f"   Updated CrowdStrike phishing prevention policies")

    # Update Shuffle workflows
    shuffle.update_phishing_workflows(incident_report)
    print(f"   Updated Shuffle phishing response workflows")
except Exception as e:
    print(f"   Security controls update failed: {e}")

# Close incident
print(f"\n[ACTION] Closing incident...")
try:
    closure_summary = {
        'closure_time': post_incident_time,
        'incident_status': 'resolved',
        'follow_up_actions': [
            'Monitor for similar phishing campaigns',
            'Conduct user security awareness training',
            'Review and update email security policies',
            'Schedule phishing simulation exercises'
        ],
        'metrics': {
            'time_to_detect': 'Calculated from detection_time',
            'time_to_contain': 'Calculated from containment_time',
            'time_to_eradicate': 'Calculated from eradication_time',
            'time_to_recover': 'Calculated from recovery_time',
            'affected_assets': len(affected_systems)
        }
    }

    iris.close_case(incident_id, closure_summary)
    print(f"   IRIS case {incident_id} closed successfully")
except Exception as e:
    print(f"   Incident closure failed: {e}")

print(f"\n Post-incident activities complete:")
print(f"  - Incident report generated and shared")
print(f"  - Lessons learned documented")
print(f"  - Security controls updated")
print(f"  - Incident case closed")

print(f"\n Phishing attack incident response completed successfully!")
print(f"   Incident ID: {incident_id}")
print(f"   Total duration: {len(phishing_indicators)} phishing indicators mitigated")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
